# TCGA → Google Drive (runtime solo CPU)

Prima fase dell'estrazione TCGA (docs/mvp.md): scarica da GDC i vetrini di un lotto del manifest `data/manifests/tcga.csv` in una cartella di Drive, con l'md5 controllato. La seconda fase, `tcga_extract_colab.ipynb` su GPU, li legge da lì: la GPU non resta ferma ad aspettare la rete.

- **Lotti.** 1 = controllore e rene (216 vetrini, ~147 GB); 2 = testa polmone (176 vetrini, ~115 GB). Il lotto 2 si scarica mentre il lotto 1 si embedda; l'estrazione cancella da Drive ogni vetrino dopo l'embedding.
- **Riprendibile.** Un vetrino entra nella cartella solo dopo il controllo dell'md5: rieseguendo le celle si riparte dai mancanti, e i vetrini già embeddati non si riscaricano.
- **Velocità.** Da GDC una connessione fa ~4 MB/s (misurato dal Mac il 25/10/2026) e i download in parallelo si sommano: `WORKERS` decide la velocità.
- **Solo dati pubblici.**

**Prima di iniziare:** Runtime → Cambia tipo di runtime → **CPU**. Drive deve avere spazio per il lotto più ~20 GB.

In [ ]:
# 1. Google Drive: vetrini, embedding, tile e QC finiscono qui, così sopravvivono alle disconnessioni
import shutil
from pathlib import Path
from google.colab import drive
drive.mount("/content/drive")
OUT = Path("/content/drive/MyDrive/tumordetector/tcga")      # come data/tcga sul Mac
RAW = Path("/content/drive/MyDrive/tumordetector/raw/tcga")  # i vetrini scaricati da td task=fetch
OUT.mkdir(parents=True, exist_ok=True)
RAW.mkdir(parents=True, exist_ok=True)
print(f"spazio libero su Drive: {shutil.disk_usage(OUT).free / 1e9:.0f} GB")

In [ ]:
# 2. Codice (se il repository è privato, clonalo con un token o rendilo pubblico)
REPO_URL = "https://github.com/masnarivan/TumorDetector.git"
REV = "origin/main"    # o un commit preciso; origin/ perché alla seconda sessione il clone esiste già e va aggiornato
!git clone -q {REPO_URL} /content/TumorDetector 2>/dev/null || git -C /content/TumorDetector fetch -q
!git -C /content/TumorDetector checkout -q {REV}
# Il progetto dichiara Python 3.12; Colab può essere più avanti (3.13 a ottobre 2026)
!pip install -q --ignore-requires-python -e /content/TumorDetector
# Le librerie che decidono maschera, griglia e pixel delle tile alle versioni di uv.lock: con quelle
# preinstallate su Colab la griglia può cambiare (e scikit-image < 0.26 non ha max_size)
!pip install -q scikit-image==0.26.0 scipy==1.18.1 pillow==12.3.0 shapely==2.2.0 \
    openslide-python==1.4.6 openslide-bin==4.0.1.2 timm==1.0.30
%cd /content/TumorDetector

In [ ]:
# 3. Download del lotto. backbone serve solo a saltare i vetrini già embeddati con H-optimus-0
BATCH = 1
WORKERS = 16
!td task=fetch data=tcga backbone=h_optimus_0 fetch.batch={BATCH} fetch.workers={WORKERS} \
    data.raw_dir={RAW} paths.out={OUT}

In [ ]:
# 4. Stato
import pandas as pd
m = pd.read_csv("data/manifests/tcga.csv")
for b, g in m.groupby("batch"):
    here = sum((RAW / f"{s}.svs").exists() for s in g.slide_id)
    done = sum((OUT / "embeddings" / "h_optimus_0" / f"{s}.npy").exists() for s in g.slide_id)
    print(f"lotto {b}: {len(g)} vetrini, {here} su Drive, {done} già embeddati")
print(f"spazio libero su Drive: {shutil.disk_usage(OUT).free / 1e9:.0f} GB")